In [8]:
import os
if 'experiments' in os.getcwd (): os.chdir (os.getcwd () + "/..")

####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet_fold_full_ensemble.csv"

######## MOOOOO ##########
SEED = int ("".join (str (ord (c)) for c in
            """  __________________
               < GORDOBOB GOT MILK! >
                 ------------------
                         \   ^__^
                          \  (oo)\________
                             (__)\        )\
                                 ||----w-||
                                 ||    ; ||
                      so much milk -> """)) % (2 ** 32)

# Preprocessing

In [9]:
TARGET_FEATURE =            "Milk_Yield_L"

DROP_FEATURES =            ["Cattle_ID",
                            "Farm_ID",
                            "Feed_Quantity_lb",
                            "Climate_Zone",
                            "Management_System",
                            "Feed_Type",
                            "Feeding_Frequency",
                            "Walking_Distance_km",
                            "Grazing_Duration_hrs",
                            "Rumination_Time_hrs",
                            "Resting_Hours",
                            "Humidity_percent",
                            "BVD_Vaccine",
                            "FMD_Vaccine",
                            "Brucellosis_Vaccine",
                            "HS_Vaccine",
                            "BQ_Vaccine",
                            "Housing_Score",
                            "Body_Condition_Score",
                            "Milking_Interval_hrs",
                            "Breed",
                            "Date"]

CATEGORICAL_FEATURES =     ["Season",
                            "Young",
                            # "DaysInMilk200",
                            "Lactation_Stage",
                            "IBR_Vaccine",
                            "Anthrax_Vaccine",
                            "Rabies_Vaccine"]

STANDARD_SCALED_FEATURES = ["Feed_Quantity_kg",
                            "Water_Intake_L",
                            "Parity",
                            "Ambient_Temperature_C",
                            "Previous_Week_Avg_Yield",
                            "Days_in_Milk",
                            "Age_Months",
                            "Weight_kg"]

In [10]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

def preprocess (
    dtrain, dtest, scaler = None
) -> tuple[pd.DataFrame, pd.DataFrame, StandardScaler]:
    """
    NOTES:
    - Interaction features do not help
    - Squaring feed for outlier overexageratting does not help
    - clipping negative records worsens results
    - Predictive and mean imputation worsen results
    - Robust & min-max scalers worsen results
    """
    ### CONVERT MONTH TO SEASON ###
    def month_to_season (m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    months = pd.to_datetime (dtest['Date']).dt.month
    dtest['Season'] = months.apply (month_to_season)

    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain['Season'] = months.apply (month_to_season)

    ### EXTRACT AGE/YOUTH PATTERN ###
    dtrain['Young'] = (dtrain['Age_Months'] < 60).astype (int)
    dtest['Young'] = (dtest['Age_Months'] < 60).astype (int)
    dtrain['Age_Months'] = np.where(dtrain['Age_Months'] < 60,
                            75 - dtrain['Age_Months'],
                            dtrain['Age_Months'])
    dtest['Age_Months'] = np.where(dtest['Age_Months'] < 60,
                            75 - dtest['Age_Months'],
                            dtest['Age_Months'])

    ### EXTRACT DAYS_IN_MILK PATTERN ###
    # dtrain['DaysInMilk200'] = (dtrain['Days_in_Milk'] < 200).astype(int)
    # dtest['DaysInMilk200'] = (dtest['Days_in_Milk'] < 200).astype(int)


    ### IMPUT MISSING FEED ###
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val

    ### ONE-HOT ENCODE CATEGORICALS ###
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES,
                             drop_first = True)
    dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES,
                            drop_first = True)
    dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    ### STANDARDIZE DATA ###
    if scaler is None:
        scaler = StandardScaler ()
        dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (dtrain[STANDARD_SCALED_FEATURES])
    else:
        dtrain[STANDARD_SCALED_FEATURES] = scaler.transform (dtrain[STANDARD_SCALED_FEATURES])

    dtest[STANDARD_SCALED_FEATURES] = scaler.transform (dtest[STANDARD_SCALED_FEATURES])

    ### DROP ALL THE USELESS FEATURES ###
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    dtest = dtest.drop (DROP_FEATURES, axis = 1)

    ### DONE ###
    return dtrain, dtest, scaler

# Training Set Eval

In [11]:
from sklearn.neural_network import MLPRegressor

ITERATIONS_PER_STEP = 10
model_template = MLPRegressor (hidden_layer_sizes = (110, 110, 110),
                               activation = "tanh",
                               learning_rate_init = 0.00003,
                               learning_rate = "adaptive",
                               early_stopping = False,
                               n_iter_no_change = 20,
                               verbose = False,
                               warm_start = True,
                               max_iter = ITERATIONS_PER_STEP,
                               random_state = SEED)

In [12]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings ('ignore', category = ConvergenceWarning)

# Params
TOLERANCE = 0.00005

# NOTE: Dropping negative records worsens results
train_data = pd.read_csv (TRAIN_PATH)
X_train = train_data.drop (TARGET_FEATURE, axis = 1)
Y_train = train_data[TARGET_FEATURE]

kf = KFold (n_splits = 5, shuffle = True, random_state = SEED)
fold_models = []
fold_scalers = []
fold_iterations = []
fold_val_rmses = []

# Train all folds
for fold_idx, (train_idx, val_idx) in enumerate (kf.split (X_train)):
    print (f"\nTraining fold {fold_idx + 1}/5...")

    # Split fold into train/validation
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = Y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = Y_train.iloc[val_idx]

    # Preprocess fold data
    X_fold_train_proc, X_fold_val_proc, fold_scaler = \
                                            preprocess (X_fold_train.copy (),
                                                        X_fold_val.copy ())

    # Train until test RMSE stops improving meaningfully
    fold_model = clone (model_template)
    prev_fold_rmse = float ('inf')
    fold_val_rmse = 0
    fold_iters = 0

    while fold_val_rmse + TOLERANCE < prev_fold_rmse:
        if fold_val_rmse > 0:
            prev_fold_rmse = fold_val_rmse

        fold_model.fit (X_fold_train_proc, y_fold_train)

        y_fold_val_pred = fold_model.predict (X_fold_val_proc)
        fold_val_rmse = np.sqrt (mean_squared_error (y_fold_val, y_fold_val_pred))

        fold_iters += ITERATIONS_PER_STEP
        print (f"  Iteration {fold_iters}: Val RMSE = {fold_val_rmse:.5f}")

    fold_val_rmses.append (fold_val_rmse)

    print (f"  Optimal iterations for fold {fold_idx + 1}: {fold_iters}")
    fold_iterations.append (fold_iters)
    fold_models.append (fold_model)
    fold_scalers.append (fold_scaler)

print (f"\nAverage fold RMSEs: {sum (fold_val_rmses) / len (fold_val_rmses)}")


Training fold 1/5...
  Iteration 10: Val RMSE = 4.18507
  Iteration 20: Val RMSE = 4.14363
  Iteration 30: Val RMSE = 4.13582
  Iteration 40: Val RMSE = 4.13306
  Iteration 50: Val RMSE = 4.13185
  Iteration 60: Val RMSE = 4.13128
  Iteration 70: Val RMSE = 4.13100
  Iteration 80: Val RMSE = 4.13088
  Iteration 90: Val RMSE = 4.13085
  Optimal iterations for fold 1: 90

Training fold 2/5...
  Iteration 10: Val RMSE = 4.15549
  Iteration 20: Val RMSE = 4.12061
  Iteration 30: Val RMSE = 4.11464
  Iteration 40: Val RMSE = 4.11249
  Iteration 50: Val RMSE = 4.11148
  Iteration 60: Val RMSE = 4.11097
  Iteration 70: Val RMSE = 4.11072
  Iteration 80: Val RMSE = 4.11063
  Iteration 90: Val RMSE = 4.11064
  Optimal iterations for fold 2: 90

Training fold 3/5...
  Iteration 10: Val RMSE = 4.16281
  Iteration 20: Val RMSE = 4.12566
  Iteration 30: Val RMSE = 4.11793
  Iteration 40: Val RMSE = 4.11487
  Iteration 50: Val RMSE = 4.11350
  Iteration 60: Val RMSE = 4.11282
  Iteration 70: Val RM

# Final Model

In [14]:
# Build "full" model
X_test = pd.read_csv (TEST_PATH)
X_train_full, X_test_full, scaler = preprocess (X_train.copy (), X_test.copy ())

model = clone (model_template)
model.max_iter = max (fold_iterations)
model.fit (X_train_full, Y_train)

fold_models.append (model)
fold_scalers.append (scaler)

In [16]:
# Ensemble predictions
print ("Generating ensemble predictions...")
ensemble_preds = []
X_train = pd.read_csv (TRAIN_PATH).drop (TARGET_FEATURE, axis = 1)
X_test = pd.read_csv (TEST_PATH)


for fold_model, fold_scaler in zip (fold_models, fold_scalers):
    temp, X_test_proc, _ = preprocess (X_train.copy (),
                                    X_test.copy (),
                                    scaler = fold_scaler)
    fold_pred = fold_model.predict (X_test_proc)
    ensemble_preds.append (fold_pred)

    train_pred = fold_model.predict (temp)
    rmse = np.sqrt (mean_squared_error (train_pred, Y_train))
    print (f"Train RMSE: {rmse}")


y_pred_ensemble = np.mean (ensemble_preds, axis = 0)

print (f"Ensemble pred mean: {y_pred_ensemble.mean ():.5f}")
print (f"Training label mean: {Y_train.mean ():.5f}")


out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (y_pred_ensemble) + 1),
                          'Milk_Yield_L': y_pred_ensemble})
out_data.to_csv (OUT_PATH, index = False)

Generating ensemble predictions...
Train RMSE: 4.104079057005432
Train RMSE: 4.103162906511914
Train RMSE: 4.10316742448826
Train RMSE: 4.104517489352921
Train RMSE: 4.103845235538638
Train RMSE: 4.10201234429143
Ensemble pred mean: 15.63739
Training label mean: 15.58916
